# Cluster Application Goes Here

In [ ]:
import pandas as pd

import  numpy as np

import  matplotlib.pyplot as plt

import seaborn as sbn

from sklearn.preprocessing import  StandardScaler

from sklearn.cluster import KMeans

from sklearn.decomposition import PCA



data = pd.read_csv("wheatseeds (1).csv")
data

# Rename columns (if not already done)
data = data.rename(columns={
    'Area': 'Area',
    'Perimeter': 'Perimeter',
    'Compactness': 'Compactness',
    'Kernel.Length': 'K_length',
    'Kernel.Width': 'K_width',
    'Asymmetry.Coeff':'A_coeff',
    'Kernel.Groove': 'K_groove'
})

# Feature selection
x = data[['Area', 'K_length']]

# Scaling
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_scaled = pd.DataFrame(x_scaled, columns=x.columns)

# Initial KMeans fit (default n_clusters)
k_model = KMeans()
k_model.fit(x_scaled)
k_labels = k_model.labels_
k_centers = k_model.cluster_centers_
k_centers = pd.DataFrame(k_centers, columns=['x', 'y'])

# PCA and Kaiser's rule
pca = PCA()
pca.fit(x_scaled)
eigen_values = pca.explained_variance_
kaiser_clusters = np.sum(eigen_values > 1)
print("Kaiser's rule suggests", kaiser_clusters, "clusters")

# Plot eigen values
plt.figure()
plt.plot(range(1, len(eigen_values) + 1), eigen_values, marker='o')
plt.axhline(y=1, color="r", linestyle="--")
plt.title("Eigenvalues")
plt.xlabel("Component")
plt.ylabel("Eigenvalue")
plt.show()

# Elbow method (inertia)
inertia = []
k_values = range(1, 11)
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(x_scaled)
    inertia.append(km.inertia_)

plt.figure()
plt.plot(list(k_values), inertia, marker='o')
plt.title("Elbow Method")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.show()

# Scatter of scaled features
plt.figure()
plt.scatter(x_scaled.Area, x_scaled.K_length)
plt.title("Scaled Features")
plt.xlabel("Area (scaled)")
plt.ylabel("K_length (scaled)")
plt.show()

# (As in original notebook) Fit KMeans with n_clusters=1 and predict clusters
k_model = KMeans(n_clusters=1, random_state=42)
k_model.fit(x_scaled)
clusters = k_model.predict(x_scaled)
data['Clusters'] = clusters

# Show columns and final cluster visualization (using earlier k_labels and k_centers)
print(data.columns)
plt.figure(figsize=(8, 6))
plt.scatter(x_scaled.Area, x_scaled.K_length, c=k_labels, cmap="viridis")
plt.scatter(k_centers.x, k_centers.y, marker="x", color="red")
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title("Observed Clusters with Cluster Centers")
plt.show()